In [ ]:
# ==============================
# Hyperparameter Optimization for Sa-Sz case
#
# Tuning the hyperparameter of the RF model for output: {Sa,Sz}
# ==============================

import h5py
import numpy as np
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

# ==============================
# Save directory
# ==============================
SAVE_DIR = "your_directory"
os.makedirs(SAVE_DIR, exist_ok=True)

# ==============================
# Load dataset
# ==============================
file_path = "your_directory"

with h5py.File(file_path, "r") as f:
    X = f["X"][:].astype(np.float32)
    Y = f["Y"][:].astype(np.float32)

print("Dataset loaded")
print("X shape:", X.shape)
print("Y shape:", Y.shape)

# ==============================
# Normalize
# ==============================
X_mean = X.mean(axis=0)
X_std = X.std(axis=0) + 1e-8
X = (X - X_mean) / X_std

Y_mean = Y.mean(axis=0)
Y_std = Y.std(axis=0) + 1e-8
Y = (Y - Y_mean) / Y_std

np.save(f"{SAVE_DIR}/X_mean_rf.npy", X_mean)
np.save(f"{SAVE_DIR}/X_std_rf.npy", X_std)
np.save(f"{SAVE_DIR}/Y_mean_rf.npy", Y_mean)
np.save(f"{SAVE_DIR}/Y_std_rf.npy", Y_std)

# ==============================
# Split
# ==============================
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# ==============================
# Objective function for Optuna
# ==============================
def objective(trial):
    # Hyperparameter search space
    n_estimators = trial.suggest_int("n_estimators", 100, 1000)
    max_depth = trial.suggest_int("max_depth", 5, 50)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Train one multi-output Random Forest
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, Y_train)
    preds = rf.predict(X_val)

    # Mean squared error across both outputs
    val_loss = np.mean((preds - Y_val) ** 2)
    return val_loss

# ==============================
# Run Optuna
# ==============================
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=50)

print("\nBEST PARAMETERS")
print(study.best_params)
np.save(f"{SAVE_DIR}/best_params_rf.npy", study.best_params)

# Save trials table
df = study.trials_dataframe()
df.to_csv(f"{SAVE_DIR}/optuna_trials_rf.csv", index=False)

# ==============================
# Plot 1: Convergence
# ==============================
vals = [t.value for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
plt.figure()
plt.plot(vals, marker='o', color='green')
plt.xlabel("Trial")
plt.ylabel("Validation Loss (MSE)")
plt.title("Optuna Convergence - Random Forest")
plt.grid(True)
plt.savefig(f"{SAVE_DIR}/convergence_rf.png", dpi=300, bbox_inches='tight')
plt.close()

# ==============================
# Plot 2: Hyperparameter Importance
# ==============================
importance = optuna.importance.get_param_importances(study)
plt.figure()
plt.bar(importance.keys(), importance.values(), color='green')
plt.ylabel("Importance")
plt.title("Hyperparameter Importance - Random Forest")
plt.savefig(f"{SAVE_DIR}/importance_rf.png", dpi=300, bbox_inches='tight')
plt.close()

# ==============================
# Plot 3: Heatmap
# ==============================
importance_df = pd.DataFrame(list(importance.items()), columns=['Parameter', 'Importance'])
importance_df = importance_df.set_index('Parameter').T
plt.figure(figsize=(max(6, len(importance)*0.6), 2))
sns.heatmap(importance_df, annot=True, cmap='viridis', vmin=0, vmax=1, cbar=True, fmt=".2f")
plt.title("Hyperparameter Importance Heatmap - Random Forest")
plt.yticks([])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/importance_heatmap_rf.png", dpi=300)
plt.close()

In [ ]:
# ==============================
# RF model for Sa-Sz case
#
# Training the RF model for output: {Sa,Sz}
# ==============================

import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import os

SAVE_DIR = "your_directory"
# Load best Optuna Random Forest parameters
best_params = np.load(f"{SAVE_DIR}/best_params_rf.npy", allow_pickle=True).item()
print("Best Random Forest Parameters:", best_params)

# ------------------------------
# Train Multi-Output Random Forest
# ------------------------------
rf = RandomForestRegressor(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    min_samples_leaf=best_params["min_samples_leaf"],
    max_features=best_params["max_features"],
    bootstrap=best_params["bootstrap"],
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, Y_train)

# ------------------------------
# Predictions
# ------------------------------
train_pred = rf.predict(X_train)
val_pred = rf.predict(X_val)

# ------------------------------
# Weighted MSE (Sa, Sz)
# ------------------------------
weights = np.array([1.0, 0.2])

train_mse_per_output = np.mean((train_pred - Y_train) ** 2, axis=0)
val_mse_per_output = np.mean((val_pred - Y_val) ** 2, axis=0)

weighted_train_loss = np.sum(train_mse_per_output * weights)
weighted_val_loss = np.sum(val_mse_per_output * weights)

print(f"Train MSE per output: {train_mse_per_output}")
print(f"Val MSE per output: {val_mse_per_output}")
print(f"Weighted Train MSE: {weighted_train_loss:.5f}")
print(f"Weighted Val MSE: {weighted_val_loss:.5f}")

# ------------------------------
# Save single multi-output model
# ------------------------------
joblib.dump(rf, f"{SAVE_DIR}/rf_model_multioutput.joblib")

# ------------------------------
# Plot Performance
# ------------------------------
plt.figure()
plt.bar(['Train', 'Validation'], [weighted_train_loss, weighted_val_loss], color='green')
plt.ylabel("Weighted MSE ($S_a$ + $S_z$)")
plt.title("Random Forest Performance (Multi-Output)")
plt.grid(axis='y')
plt.savefig(f"{SAVE_DIR}/learning_curve_rf.png", dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# ==============================
# Inference of the RF model for Sa-Sz case
#
# Inference of the RF model for output: {Sa,Sz}
# ==============================

import numpy as np
import matplotlib.pyplot as plt
import h5py
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
import joblib
import os

# ==============================
# PATH
# ==============================
SAVE_DIR = "your_directory"

# ==============================
# Load dataset
# ==============================
file_path = f"{SAVE_DIR}/dataset_large.h5"
with h5py.File(file_path, "r") as f:
    X = f["X"][:].astype(np.float32)
    Y = f["Y"][:].astype(np.float32)

print("Dataset loaded")
print("X shape:", X.shape)
print("Y shape:", Y.shape)

# ==============================
# Load normalization
# ==============================
X_mean = np.load(f"{SAVE_DIR}/X_mean_rf.npy")
X_std  = np.load(f"{SAVE_DIR}/X_std_rf.npy")
Y_mean = np.load(f"{SAVE_DIR}/Y_mean_rf.npy")
Y_std  = np.load(f"{SAVE_DIR}/Y_std_rf.npy")

X_norm = (X - X_mean) / X_std

# ==============================
# Split dataset
# ==============================
X_train, X_val_norm, Y_train, Y_val = train_test_split(
    X_norm, Y, test_size=0.2, random_state=42
)

# ==============================
# Load multi-output RF model
# ==============================
rf = joblib.load(f"{SAVE_DIR}/rf_model_multioutput.joblib")

# ==============================
# Predictions
# ==============================
preds_norm = rf.predict(X_val_norm)   # shape: (N, 2)

# Denormalize
Y_pred = preds_norm * Y_std + Y_mean

# ==============================
# Metrics
# ==============================
weights = np.array([1.0, 0.2])

for i, name in enumerate(["Sa", "Sz"]):
    rmse_std = np.sqrt(mean_squared_error(
        (Y_val[:, i] - Y_mean[i]) / Y_std[i],
        preds_norm[:, i]
    ))
    rmse = np.sqrt(mean_squared_error(Y_val[:, i], Y_pred[:, i]))
    r2 = r2_score(Y_val[:, i], Y_pred[:, i])

    print(f"{name} -> R2: {r2:.4f}, RMSE: {rmse:.4f}, RMSE (std): {rmse_std:.4f}")

# ==============================
# Parity plots
# ==============================
def parity_plot(y_true, y_pred, name, save_path, xlim=None, ylim=None, color='green'):
    plt.figure(figsize=(5,5))
    plt.scatter(y_true, y_pred, alpha=0.6, color=color, s=5)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    plt.xlabel(rf"Actual ${name}$ ($\mu$m)")
    plt.ylabel(rf"Predicted ${name}$ ($\mu$m)")
    plt.title(f"Parity Plot - ${name}$")
    plt.grid(True)
    if xlim: plt.xlim(xlim)
    if ylim: plt.ylim(ylim)
    plt.axis('equal')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

parity_plot(Y_val[:,0], Y_pred[:,0], "S_a", f"{SAVE_DIR}/parity_Sa_rf.png", xlim=(0,8), ylim=(0,8), color='blue')
parity_plot(Y_val[:,1], Y_pred[:,1], "S_z", f"{SAVE_DIR}/parity_Sz_rf.png", xlim=(0,50), ylim=(0,50), color='green')

# ==============================
# Feature Sensitivity (Permutation)
# ==============================
feature_names = ["vc","fz","eps_r","eps_a","z","ri"]
output_names  = ["Sa","Sz"]

baseline_loss = ((Y_val - Y_pred)**2).mean(axis=0)
importance_matrix = np.zeros((2, X_val_norm.shape[1]))

for j in range(X_val_norm.shape[1]):
    X_perm = X_val_norm.copy()
    np.random.shuffle(X_perm[:, j])

    perm_preds_norm = rf.predict(X_perm)
    perm_preds = perm_preds_norm * Y_std + Y_mean

    loss = ((Y_val - perm_preds)**2).mean(axis=0)
    importance_matrix[:, j] = loss - baseline_loss

# Normalize importance
importance_matrix_norm = importance_matrix / importance_matrix.max()

# ==============================
# Heatmap
# ==============================
plt.figure(figsize=(8,3))
im = plt.imshow(importance_matrix_norm, aspect='auto', cmap='Greens', vmin=0, vmax=1)
plt.colorbar(im, label="Normalized ΔMSE")

for i in range(importance_matrix_norm.shape[0]):
    for j in range(importance_matrix_norm.shape[1]):
        plt.text(j, i, f"{importance_matrix_norm[i,j]:.2f}",
                 ha='center', va='center', color='black', fontsize=9)

plt.xticks(range(len(feature_names)), feature_names, rotation=30)
plt.yticks(range(len(output_names)), output_names)
plt.title("Feature Sensitivity Matrix (Normalized) - RF")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/feature_importance_matrix_norm_rf.png", dpi=300)
plt.close()

In [ ]:
# ==============================
# Inverse design of RF with DIVERSITY + GLOBAL EXPLORATION for output: {Sa,Sz}
# ==============================

import numpy as np
import joblib
import optuna
import matplotlib.pyplot as plt
import pandas as pd

# ------------------------------
# SETTINGS
# ------------------------------
SAVE_DIR = "your_directory"

target_Sa = 6
target_Sz = 33

n_trials = 200   # per (z, ri)
top_k = 10       # per (z, ri)
lambda_div = 0.1 # diversity strength

# ------------------------------
# LOAD NORMALIZATION
# ------------------------------
X_mean = np.load(f"{SAVE_DIR}/X_mean.npy")
X_std  = np.load(f"{SAVE_DIR}/X_std.npy")
Y_mean = np.load(f"{SAVE_DIR}/Y_mean.npy")
Y_std  = np.load(f"{SAVE_DIR}/Y_std.npy")

# ------------------------------
# LOAD RANDOM FOREST MODEL
# ------------------------------
rf = joblib.load(f"{SAVE_DIR}/rf_model_multioutput.joblib")

# ------------------------------
# LOSS WEIGHTS
# ------------------------------
w_Sa = 1.0
w_Sz = 0.2
# w_Sz = 1.0

# ------------------------------
# DIVERSITY STORAGE
# ------------------------------
visited_points = []

def diversity_penalty(x):
    if len(visited_points) == 0:
        return 0.0
    dists = [np.linalg.norm(x - vp) for vp in visited_points]
    return np.exp(-min(dists))  # higher if closer

# ------------------------------
# STORAGE FOR ALL RESULTS
# ------------------------------
all_results = []

# ------------------------------
# LOOP OVER DISCRETE VARIABLES
# ------------------------------
for z_fixed in [2.0, 3.0, 4.0]:
    for ri_fixed in [3.0, 4.0, 5.0]:

        print(f"\n=== Optimizing for z={z_fixed}, ri={ri_fixed} ===")
        visited_points = []

        def objective(trial):
            vc    = trial.suggest_float("vc", 100, 300)
            fz    = trial.suggest_float("fz", 0.1, 0.9)
            eps_r = trial.suggest_float("eps_r", -0.026, 0.011)
            eps_a = trial.suggest_float("eps_a", 0.003, 0.009)

            x = np.array([[vc, fz, eps_r, eps_a, z_fixed, ri_fixed]], dtype=np.float32)
            x_norm = (x - X_mean) / X_std

            pred = rf.predict(x_norm)[0]  # Random Forest prediction
            y_pred = pred * Y_std + Y_mean
            Sa_pred, Sz_pred = y_pred

            # Main loss
            Sa_err = (Sa_pred - target_Sa) / Y_std[0]
            Sz_err = (Sz_pred - target_Sz) / Y_std[1]
            loss = w_Sa * Sa_err**2 + w_Sz * Sz_err**2

            # Diversity penalty
            x_raw = np.array([vc, fz, eps_r, eps_a, z_fixed, ri_fixed])
            loss += lambda_div * diversity_penalty(x_raw)

            return loss

        # ------------------------------
        # OPTUNA SAMPLER
        # ------------------------------
        sampler = optuna.samplers.TPESampler(
            n_startup_trials=100,
            multivariate=True,
            group=True,
            seed=42
        )

        study = optuna.create_study(direction="minimize", sampler=sampler)
        study.optimize(objective, n_trials=n_trials)

        df = study.trials_dataframe()
        df_sorted = df.sort_values("value").reset_index(drop=True)

        # ------------------------------
        # STORE TOP-K PER CONFIG
        # ------------------------------
        for i in range(top_k):
            row = df_sorted.iloc[i]
            x = np.array([[row['params_vc'], row['params_fz'],
                           row['params_eps_r'], row['params_eps_a'],
                           z_fixed, ri_fixed]], dtype=np.float32)
            x_norm = (x - X_mean) / X_std
            pred = rf.predict(x_norm)[0]
            y_pred = pred * Y_std + Y_mean

            all_results.append({
                "z": z_fixed,
                "ri": ri_fixed,
                "vc": row['params_vc'],
                "fz": row['params_fz'],
                "eps_r": row['params_eps_r'],
                "eps_a": row['params_eps_a'],
                "Sa": y_pred[0],
                "Sz": y_pred[1],
                "loss": row['value']
            })

# ------------------------------
# FINAL RESULTS
# ------------------------------
df_all = pd.DataFrame(all_results)
df_all = df_all.sort_values("loss").reset_index(drop=True)
df_all.to_csv(f"{SAVE_DIR}/inverse_diverse_solutions_rf.csv", index=False)
print("\n=== FINAL DIVERSE SOLUTIONS (RF) ===\n")
print(df_all.head(20))

# ------------------------------
# TOP-K TABLE FIGURE
# ------------------------------
top_table = df_all.head(20).copy()
top_table.insert(0, "No.", range(1, len(top_table)+1))

# Format columns
top_table["vc"]    = top_table["vc"].map("{:.2f}".format)
top_table["fz"]    = top_table["fz"].map("{:.3f}".format)
top_table["eps_r"] = top_table["eps_r"].map("{:.4f}".format)
top_table["eps_a"] = top_table["eps_a"].map("{:.4f}".format)
top_table["Sa"]    = top_table["Sa"].map("{:.2f}".format)
top_table["Sz"]    = top_table["Sz"].map("{:.2f}".format)
top_table["loss"]  = top_table["loss"].map("{:.4f}".format)
top_table["z"]     = top_table["z"].map("{:.0f}".format)
top_table["ri"]    = top_table["ri"].map("{:.0f}".format)

top_table = top_table.rename(columns={
    "vc": r"$v_c$", "fz": r"$f_z$", "eps_r": r"$\epsilon_r$", "eps_a": r"$\epsilon_a$",
    "Sa": r"$S_a$", "Sz": r"$S_z$", "z": r"$z$", "ri": r"$r_i$", "loss": "Loss"
})

fig, ax = plt.subplots(figsize=(14,6))
ax.axis('off')
tbl = ax.table(cellText=top_table.values, colLabels=top_table.columns, cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)

col_widths = [0.03,0.03,0.03,0.06,0.06,0.06,0.06,0.06,0.06,0.06]
for i, width in enumerate(col_widths):
    for key, cell in tbl.get_celld().items():
        if key[1] == i:
            cell.set_width(width)

plt.title("(RF) Top-20 Inverse Design Solutions (RF)", fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/top_solutions_table_rf.png", dpi=300)
plt.show()

# ------------------------------
# PLOT DIVERSE SOLUTIONS WITH COLORBAR
# ------------------------------
plt.figure(figsize=(6,5))
df_all["config_id"] = df_all.groupby(["z", "ri"]).ngroup()
scatter = plt.scatter(df_all["Sa"], df_all["Sz"], c=df_all["config_id"], cmap="tab10", alpha=0.8)
plt.scatter(target_Sa, target_Sz, color='black', s=100, label="Target")
cbar = plt.colorbar(scatter)
cbar.set_label("Configuration (z, $r_i$)")
configs = df_all[["config_id","z","ri"]].drop_duplicates().sort_values("config_id")
cbar.set_ticks(configs["config_id"])
cbar.set_ticklabels([f"z={int(z)}, $r_i$={int(ri)}" for z,ri in zip(configs["z"], configs["ri"])])
plt.xlabel("$S_a$ ($\mu$m)")
plt.ylabel("$S_z$ ($\mu$m)")
plt.title("(RF) Inverse Design Solution Space")
plt.legend()
plt.grid(True)
plt.savefig(f"{SAVE_DIR}/inverse_diverse_solution_space_rf.png", dpi=300)
plt.show()



# ==============================
# PAPER-READY VISUALIZATION BLOCK (RANDOM FOREST)
# ==============================

import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------
# 1. CONVERGENCE PLOTS
# ------------------------------

# Assuming 'study' is the Optuna study for RF inverse design
values = [t.value for t in study.trials if t.value is not None]

plt.figure(figsize=(6,4))
plt.plot(values, alpha=0.7, color='green')
plt.xlabel("Trial")
plt.ylabel("Loss")
plt.title("(RF) Inverse Design Convergence")
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/rf_convergence.png", dpi=300)
plt.show()

# ------------------------------
# 2. Solution Space Plot (Sa vs Sz)
# ------------------------------

# Assign a unique id per (z, ri) combination
df_all["config_id"] = df_all.groupby(["z", "ri"]).ngroup()

plt.figure(figsize=(6,5))
scatter = plt.scatter(
    df_all["Sa"],
    df_all["Sz"],
    c=df_all["config_id"],
    cmap="tab10",
    alpha=0.8
)

# Target point in black
plt.scatter(target_Sa, target_Sz, color='black', s=100, label="Target")

# Colorbar for configuration
cbar = plt.colorbar(scatter)
cbar.set_label("Configuration (z, r_i)")

# Custom ticks showing (z, ri)
configs = df_all[["config_id", "z", "ri"]].drop_duplicates().sort_values("config_id")
cbar.set_ticks(configs["config_id"])
cbar.set_ticklabels([f"z={int(z)}, r_i={int(ri)}" for z, ri in zip(configs["z"], configs["ri"])])

plt.xlabel("Sa (μm)")
plt.ylabel("Sz (μm)")
plt.title("(RF) Inverse Design Solution Space")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/rf_inverse_solution_space.png", dpi=300)
plt.show()

In [ ]:
# ==============================
# Hyperparameter Optimization for Sa-only case
#
# Tuning the hyperparameter of the RF model for output: {Sa}
# ==============================

import h5py
import numpy as np
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

# ==============================
# Save directory
# ==============================
SAVE_DIR = "your_directory"
os.makedirs(SAVE_DIR, exist_ok=True)

# ==============================
# Load dataset
# ==============================
file_path = "your_directory"

with h5py.File(file_path, "r") as f:
    X = f["X"][:].astype(np.float32)
    Y = f["Y"][:].astype(np.float32)

print("Dataset loaded")
print("X shape:", X.shape)
print("Y shape:", Y.shape)

# ==============================
# Use ONLY Sa (first column)
# ==============================
Y = Y[:, 0]   # shape becomes (N,)

# ==============================
# Normalize
# ==============================
X_mean = X.mean(axis=0)
X_std = X.std(axis=0) + 1e-8
X = (X - X_mean) / X_std

Y_mean = Y.mean()
Y_std = Y.std() + 1e-8
Y = (Y - Y_mean) / Y_std

# Save normalization (NEW names)
np.save(f"{SAVE_DIR}/X_mean_rf_sa.npy", X_mean)
np.save(f"{SAVE_DIR}/X_std_rf_sa.npy", X_std)
np.save(f"{SAVE_DIR}/Y_mean_rf_sa.npy", Y_mean)
np.save(f"{SAVE_DIR}/Y_std_rf_sa.npy", Y_std)

# ==============================
# Split
# ==============================
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# ==============================
# Objective function
# ==============================
def objective(trial):

    n_estimators = trial.suggest_int("n_estimators", 100, 1000)
    max_depth = trial.suggest_int("max_depth", 5, 50)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, Y_train)
    preds = rf.predict(X_val)

    mse = np.mean((preds - Y_val) ** 2)
    return mse

# ==============================
# Run Optuna
# ==============================
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=50)

print("\nBEST PARAMETERS")
print(study.best_params)

# Save best params (NEW name)
np.save(f"{SAVE_DIR}/best_params_rf_sa.npy", study.best_params)

# Save trials
df = study.trials_dataframe()
df.to_csv(f"{SAVE_DIR}/optuna_trials_rf_sa.csv", index=False)

# ==============================
# Plot 1: Convergence
# ==============================
vals = [t.value for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

plt.figure()
plt.plot(vals, marker='o', color='green')
plt.xlabel("Trial")
plt.ylabel("Validation Loss (MSE)")
plt.title("Optuna Convergence - RF ($S_a$)")
plt.grid(True)
plt.savefig(f"{SAVE_DIR}/convergence_rf_sa.png", dpi=300, bbox_inches='tight')
plt.close()

# ==============================
# Plot 2: Hyperparameter Importance
# ==============================
importance = optuna.importance.get_param_importances(study)

plt.figure()
plt.bar(importance.keys(), importance.values(), color='green')
plt.ylabel("Importance")
plt.title("Hyperparameter Importance - RF ($S_a$)")
plt.savefig(f"{SAVE_DIR}/importance_rf_sa.png", dpi=300, bbox_inches='tight')
plt.close()

# ==============================
# Plot 3: Heatmap
# ==============================
importance_df = pd.DataFrame(list(importance.items()), columns=['Parameter', 'Importance'])
importance_df = importance_df.set_index('Parameter').T

plt.figure(figsize=(max(6, len(importance)*0.6), 2))
sns.heatmap(importance_df, annot=True, cmap='viridis', vmin=0, vmax=1, cbar=True, fmt=".2f")

plt.title("Hyperparameter Importance Heatmap - RF (Sa)")
plt.yticks([])
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/importance_heatmap_rf_sa.png", dpi=300)
plt.close()

In [ ]:
# ==============================
# FINAL MODEL TRAINING - Sa only
#
# Training the RF model for output: {Sa}
# ==============================

import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import os

SAVE_DIR = "your_directory"

# ==============================
# Load best Optuna parameters (Sa)
# ==============================
best_params = np.load(f"{SAVE_DIR}/best_params_rf_sa.npy", allow_pickle=True).item()
print("Best Random Forest Parameters (Sa):", best_params)

# ==============================
# Train Random Forest (single output)
# ==============================
rf = RandomForestRegressor(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    min_samples_leaf=best_params["min_samples_leaf"],
    max_features=best_params["max_features"],
    bootstrap=best_params["bootstrap"],
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, Y_train)

# ==============================
# Evaluate
# ==============================
train_pred = rf.predict(X_train)
val_pred   = rf.predict(X_val)

train_loss = mean_squared_error(Y_train, train_pred)
val_loss   = mean_squared_error(Y_val, val_pred)

print(f"Train MSE (Sa): {train_loss:.5f}")
print(f"Val   MSE (Sa): {val_loss:.5f}")

# ==============================
# Save model
# ==============================
joblib.dump(rf, f"{SAVE_DIR}/rf_model_sa.joblib")

# ==============================
# Plot "Learning Curve" (bar)
# ==============================
plt.figure()
plt.bar(['Train', 'Validation'], [train_loss, val_loss], color='green')
plt.ylabel("MSE ($S_a$)")
plt.title("Random Forest Performance ($S_a$)")
plt.grid(axis='y')

plt.savefig(f"{SAVE_DIR}/learning_curve_rf_sa.png", dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# ==============================
# INFERENCE PIPELINE - Sa only
#
# Inference of the trained RF model for output: {Sa}
# ==============================

import numpy as np
import matplotlib.pyplot as plt
import h5py
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
import joblib
import os

# ==============================
# PATH
# ==============================
SAVE_DIR = "your_directory"

# ==============================
# Load dataset
# ==============================
file_path = f"{SAVE_DIR}/dataset_large.h5"
with h5py.File(file_path, "r") as f:
    X = f["X"][:].astype(np.float32)
    Y = f["Y"][:].astype(np.float32)

print("Dataset loaded")
print("X shape:", X.shape)
print("Y shape:", Y.shape)

# ==============================
# Use ONLY Sa
# ==============================
Y = Y[:, 0]   # shape (N,)

# ==============================
# Load normalization (Sa version)
# ==============================
X_mean = np.load(f"{SAVE_DIR}/X_mean_rf_sa.npy")
X_std  = np.load(f"{SAVE_DIR}/X_std_rf_sa.npy")
Y_mean = np.load(f"{SAVE_DIR}/Y_mean_rf_sa.npy")
Y_std  = np.load(f"{SAVE_DIR}/Y_std_rf_sa.npy")

# Normalize features
X_norm = (X - X_mean) / X_std

# ==============================
# Split dataset
# ==============================
X_train, X_val_norm, Y_train, Y_val = train_test_split(
    X_norm, Y, test_size=0.2, random_state=42
)

# ==============================
# Load trained RF model
# ==============================
rf = joblib.load(f"{SAVE_DIR}/rf_model_sa.joblib")

# ==============================
# Predictions
# ==============================
preds_norm = rf.predict(X_val_norm)

# Denormalize
Y_pred = preds_norm * Y_std + Y_mean

# ==============================
# Metrics
# ==============================
# Standardized RMSE
rmse_std = np.sqrt(mean_squared_error((Y_val - Y_mean) / Y_std, preds_norm))

# Original RMSE
rmse = np.sqrt(mean_squared_error(Y_val, Y_pred))

# R2
r2 = r2_score(Y_val, Y_pred)

print(f"Sa -> R2: {r2:.4f}, RMSE: {rmse:.4f}, RMSE (std): {rmse_std:.4f}")

# ==============================
# Parity plot
# ==============================
def parity_plot(y_true, y_pred, name, save_path, xlim=None, ylim=None, color='blue'):
    plt.figure(figsize=(5,5))
    plt.scatter(y_true, y_pred, alpha=0.6, color=color, s=5)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    plt.xlabel(rf"Actual ${name}$ ($\mu$m)")
    plt.ylabel(rf"Predicted ${name}$ ($\mu$m)")
    plt.title(f"Parity Plot - ${name}$")
    plt.grid(True)
    if xlim: plt.xlim(xlim)
    if ylim: plt.ylim(ylim)
    plt.axis('equal')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

parity_plot(Y_val, Y_pred, "S_a", f"{SAVE_DIR}/parity_Sa_rf_sa.png",
            xlim=(0,8), ylim=(0,8), color='blue')

# ==============================
# Feature Sensitivity (Permutation)
# ==============================
feature_names = ["vc","fz","eps_r","eps_a","z","ri"]

baseline_loss = np.mean((Y_val - Y_pred)**2)
importance = np.zeros(X_val_norm.shape[1])

for j in range(X_val_norm.shape[1]):
    X_perm = X_val_norm.copy()
    np.random.shuffle(X_perm[:, j])

    perm_preds_norm = rf.predict(X_perm)
    perm_preds = perm_preds_norm * Y_std + Y_mean

    loss = np.mean((Y_val - perm_preds)**2)
    importance[j] = loss - baseline_loss

# Normalize to [0,1]
importance_norm = importance / importance.max()

# ==============================
# Bar Plot (cleaner for 1 output)
# ==============================
plt.figure(figsize=(8,4))
plt.bar(feature_names, importance_norm)
plt.ylabel("Normalized ΔMSE")
plt.title("Feature Importance (Permutation) - RF (Sa)")
plt.xticks(rotation=30)
plt.grid(axis='y')

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/feature_importance_rf_sa.png", dpi=300)
plt.close()